## Data Analytics

### Chapter 20: Multiple Imputation by Chained Equation (MICE)

Multiple Imputation by Chained Equations (MICE) is a statistical technique for handling missing data by learning the relationships between variables rather than replacing missing values with a single fixed statistic.

Scikit-Learn provides an implementation of the MICE algorithm through the **IterativeImputer** class. To better understand how it improves missing value imputation, we will compare it with two simpler techniques:

- **SimpleImputer** – replaces missing values using statistical measures such as the mean.

- **KNNImputer** – estimates missing values using the nearest neighboring samples.

- **IterativeImputer (MICE)** – repeatedly trains machine learning models to predict and refine missing values by learning the relationships among variables.

For this demonstration, we will generate a synthetic dataset containing **200 samples** and randomly introduce **30% missing values**. We will then apply each imputation method and compare their results.

#### Import Required Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer

# Enable IterativeImputer (experimental feature)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

#### Generate a Dataset

In [2]:
np.random.seed(42)

n = 200

age = np.random.randint(18, 31, n)
height = 145 + age * 1.5 + np.random.normal(0, 4, n)
weight = 20 + height * 0.38 + np.random.normal(0, 5, n)
gpa = np.clip(2 + (31 - age) * 0.08 + np.random.normal(0, 0.25, n), 2.0, 4.0)

df = pd.DataFrame({
    "Age": age,
    "Height": np.round(height, 1),
    "Weight": np.round(weight, 1),
    "GPA": np.round(gpa, 2)
})

df.head(20)

,Age,Height,Weight,GPA
0,24,179.4,90.0,2.94
1,21,170.6,81.4,2.93
2,30,191.2,97.1,2.00
3,28,188.0,93.0,2.19
4,25,182.5,93.4,2.26
5,30,189.1,95.0,2.00
6,22,172.3,81.3,2.95
7,24,179.3,85.3,3.04
8,27,184.1,93.7,2.00
9,20,171.8,88.3,3.02


In [3]:
# Introduce 30% Missing Values
df_missing = df.copy()

mask = np.random.rand(*df_missing.shape) < 0.30
df_missing = df_missing.mask(mask)

df_missing.head(20)

,Age,Height,Weight,GPA
0,NaN,179.4,NaN,NaN
1,NaN,NaN,NaN,2.93
2,30.0,191.2,97.1,2.00
3,NaN,188.0,93.0,NaN
4,25.0,182.5,93.4,2.26
5,NaN,189.1,95.0,NaN
6,22.0,172.3,81.3,2.95
7,24.0,NaN,NaN,3.04
8,27.0,NaN,NaN,2.00
9,20.0,171.8,NaN,NaN


In [4]:
# Check the number of missing values.
df_missing.isna().sum()

Age       67
Height    59
Weight    64
GPA       52
dtype: int64

#### 20.1. Simple Imputation

In [5]:
simple_imputer = SimpleImputer(strategy="mean")

df_simple = pd.DataFrame(
    simple_imputer.fit_transform(df_missing),
    columns=df_missing.columns
)

df_simple.head(20)

,Age,Height,Weight,GPA
0,23.849624,179.400000,89.377206,2.512432
1,23.849624,181.895035,89.377206,2.930000
2,30.000000,191.200000,97.100000,2.000000
3,23.849624,188.000000,93.000000,2.512432
4,25.000000,182.500000,93.400000,2.260000
5,23.849624,189.100000,95.000000,2.512432
6,22.000000,172.300000,81.300000,2.950000
7,24.000000,181.895035,89.377206,3.040000
8,27.000000,181.895035,89.377206,2.000000
9,20.000000,171.800000,89.377206,2.512432


#### 20.2. KNN Imputation

In [6]:
knn_imputer = KNNImputer(n_neighbors=5)

df_knn = pd.DataFrame(
    knn_imputer.fit_transform(df_missing),
    columns=df_missing.columns
)

df_knn.head(20)

,Age,Height,Weight,GPA
0,23.0,179.40,88.50,2.616
1,20.8,174.34,81.42,2.930
2,30.0,191.20,97.10,2.000
3,25.8,188.00,93.00,2.518
4,25.0,182.50,93.40,2.260
5,26.8,189.10,95.00,2.270
6,22.0,172.30,81.30,2.950
7,24.0,180.04,86.54,3.040
8,27.0,185.94,95.00,2.000
9,20.0,171.80,87.94,3.078


#### 20.3. MICE (Iterative Imputation)

In [7]:
mice_imputer = IterativeImputer(
    random_state=42,
    max_iter=10
)

df_mice = pd.DataFrame(
    mice_imputer.fit_transform(df_missing),
    columns=df_missing.columns
)

df_mice.head(20)

,Age,Height,Weight,GPA
0,23.147022,179.400000,88.084265,2.612023
1,20.681451,176.325354,86.217943,2.930000
2,30.000000,191.200000,97.100000,2.000000
3,27.309537,188.000000,93.000000,2.268572
4,25.000000,182.500000,93.400000,2.260000
5,27.900373,189.100000,95.000000,2.234255
6,22.000000,172.300000,81.300000,2.950000
7,24.000000,181.198840,89.199855,3.040000
8,27.000000,185.840575,91.738833,2.000000
9,20.000000,171.800000,83.949358,2.870652


#### 20.4. Compare the Results

In [8]:
comparison = pd.DataFrame({
    "Original": df["Height"],
    "Missing": df_missing["Height"],
    "Simple": df_simple["Height"],
    "KNN": df_knn["Height"],
    "MICE": df_mice["Height"]
})

comparison.head(20)

,Original,Missing,Simple,KNN,MICE
0,179.4,179.4,179.400000,179.40,179.400000
1,170.6,NaN,181.895035,174.34,176.325354
2,191.2,191.2,191.200000,191.20,191.200000
3,188.0,188.0,188.000000,188.00,188.000000
4,182.5,182.5,182.500000,182.50,182.500000
5,189.1,189.1,189.100000,189.10,189.100000
6,172.3,172.3,172.300000,172.30,172.300000
7,179.3,NaN,181.895035,180.04,181.198840
8,184.1,NaN,181.895035,185.94,185.840575
9,171.8,171.8,171.800000,171.80,171.800000


In [9]:
# You can also verify that all missing values have been filled.
print(df_missing.isna().sum())
print(df_simple.isna().sum())
print(df_knn.isna().sum())
print(df_mice.isna().sum())

Age       67
Height    59
Weight    64
GPA       52
dtype: int64
Age       0
Height    0
Weight    0
GPA       0
dtype: int64
Age       0
Height    0
Weight    0
GPA       0
dtype: int64
Age       0
Height    0
Weight    0
GPA       0
dtype: int64
